# Create Final Figure 

In [1]:
from __future__ import annotations

from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.path import Path as MplPath
from matplotlib.patches import PathPatch, Rectangle


# ============================================================
# Step 1. File names
# ============================================================

PANEL_A_REGION_INPUT = Path("step2_region_category_average_shares.csv")
PANEL_B_REGION_INPUT = Path("step4_region_seea_account_average_coverage.csv")
PANEL_C_LINKS_INPUT = Path("step5_panel_C_sankey_links.csv")

# Used to make sure no_account is included in Panel C
# and to calculate Data source (n = xxxx).
# Required columns: dataset, theme, weighted_indicator_count
PANEL_C_THEME_SUMMARY_INPUT = Path("step5_source_by_seea_theme_summary.csv")

# File used to add indicator counts to Panel A x-axis labels.
# This file must include columns: category and indicator_count.
PANEL_A_COUNT_INPUT = Path("step1_indicator_category_summary.csv")

OUTPUT_PDF = Path("combined_figure_panels_A_B_C.pdf")
OUTPUT_SVG = Path("combined_figure_panels_A_B_C.svg")
OUTPUT_PNG = Path("combined_figure_panels_A_B_C.png")


# ============================================================
# Step 2. Figure settings
# ============================================================

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["font.family"] = "DejaVu Sans"
mpl.rcParams["axes.unicode_minus"] = False


# ============================================================
# Step 2b. Journal-ready font sizes
# ============================================================

TITLE_FS = 20
AXIS_FS = 13
CELL_FS = 13

SANK_TITLE_FS = 20
SANK_HEADER_FS = 13
SANK_SOURCE_FS = 15
SANK_SOURCE_VALUE_FS = 13
SANK_LABEL_FS = 13
SANK_VALUE_FS = 12.5
NOTE_FS = 12

CBAR_FS = 13


# ============================================================
# Step 3. Region order and labels
# ============================================================

REGION_ORDER = [
    "Eurostat + UK",
    "Non-Eurostat Europe & Central Asia",
    "Middle East, North Africa, Afghanistan & Pakistan",
    "Sub-Saharan Africa",
    "South Asia",
    "East Asia & Pacific",
    "North America",
    "Latin America & Caribbean",
]

REGION_LABELS = {
    "Middle East, North Africa, Afghanistan & Pakistan": "MENA,\nAfghanistan & Pakistan",
    "Latin America & Caribbean": "Latin America &\nCaribbean",
    "Eurostat + UK": "Eurostat + UK",
    "Non-Eurostat Europe & Central Asia": "Non-Eurostat Europe &\nCentral Asia",
    "East Asia & Pacific": "East Asia &\nPacific",
    "Sub-Saharan Africa": "Sub-Saharan\nAfrica",
}


# ============================================================
# Step 4. Panel A columns and labels
# ============================================================

PANEL_A_COLUMNS = [
    "Physical stock",
    "Monetary stock",
    "Physical flow",
    "Monetary flow",
    "Action_kept",
    "Policy Guide_kept",
    "HPS_monetary_only",
    "DDP_monetary_only",
    "MS-NMS_monetary_only",
    "ADE_monetary_only",
]

PANEL_A_BASE_LABELS = {
    "Physical stock": "Physical\nstock",
    "Monetary stock": "Monetary\nstock",
    "Physical flow": "Physical\nflow",
    "Monetary flow": "Monetary\nflow",
    "Action_kept": "Action",
    "Policy Guide_kept": "Policy\nguide",
    "HPS_monetary_only": "Monetary\nHPS",
    "DDP_monetary_only": "Monetary\nDDP",
    "MS-NMS_monetary_only": "Monetary\nMS-NMS",
    "ADE_monetary_only": "Monetary\nADE",
}


def make_panel_a_labels() -> dict[str, str]:
    counts = pd.read_csv(PANEL_A_COUNT_INPUT)

    required_cols = {"category", "indicator_count"}
    missing_cols = required_cols - set(counts.columns)

    if missing_cols:
        raise ValueError(
            f"{PANEL_A_COUNT_INPUT} is missing required columns: {missing_cols}"
        )

    count_map = dict(
        zip(
            counts["category"].astype(str),
            counts["indicator_count"].astype(int),
        )
    )

    return {
        col: f"{PANEL_A_BASE_LABELS[col]}\n(n={count_map.get(col, 0)})"
        for col in PANEL_A_COLUMNS
    }


PANEL_A_LABELS = make_panel_a_labels()


# ============================================================
# Step 5. Panel B columns and labels
# ============================================================

PANEL_B_COLUMNS = [
    "Reported SEEA accounts",
    "Easily accessible accounts",
    "Found but not reported",
]

PANEL_B_LABELS = {
    "Reported SEEA accounts": "Reported\nSEEA accounts",
    "Easily accessible accounts": "Easily accessible\naccounts",
    "Found but not reported": "Account found but not\nreported",
}


# ============================================================
# Step 6. Panel C Sankey settings
# ============================================================

DATASET_ORDER = ["WB", "IMF", "UNSD"]

DATASET_COLORS = {
    "WB": "#b2182b",
    "IMF": "#238b45",
    "UNSD": "#2166ac",
}

LINK_COLORS = {
    "WB": (178 / 255, 24 / 255, 43 / 255, 0.26),
    "IMF": (35 / 255, 139 / 255, 69 / 255, 0.24),
    "UNSD": (33 / 255, 102 / 255, 172 / 255, 0.25),
}

THEME_ORDER = [
    "Material flow",
    "Water",
    "Energy",
    "Mineral and energy resources",
    "Air emissions",
    "Land",
    "Timber resources",
    "Aquatic resources",
    "Other biological resources",
    "Agriculture, forestry and fisheries",
    "Waste",
    "Environmental protection expenditure",
    "Resource management expenditure",
    "Environmental goods and services",
    "Environmental taxes",
    "Environmental subsidies and similar transfers",
    "Ecosystem extent",
    "Ecosystem condition",
    "Ecosystem services",
    "Ecosystem asset",
    "Ocean",
    "Species",
    "Carbon",
    "Urban",
    "Protected areas",
    "Forest",
    "No direct SEEA account flag",
]

THEME_LABELS = {
    "Material flow": "Material flow",
    "Water": "Water",
    "Energy": "Energy",
    "Mineral and energy resources": "Mineral and energy\nresources",
    "Air emissions": "Air emissions",
    "Land": "Land",
    "Timber resources": "Timber resources",
    "Aquatic resources": "Aquatic resources",
    "Other biological resources": "Other biological resources",
    "Agriculture, forestry and fisheries": "Agriculture, forestry and fisheries",
    "Waste": "Waste",
    "Environmental protection expenditure": "Environmental protection expenditure",
    "Resource management expenditure": "Resource management expenditure",
    "Environmental goods and services": "Environmental goods and services",
    "Environmental taxes": "Environmental taxes",
    "Environmental subsidies and similar transfers": "Environmental subsidies and similar transfers",
    "Ecosystem extent": "Ecosystem extent",
    "Ecosystem condition": "Ecosystem condition",
    "Ecosystem services": "Ecosystem services",
    "Ecosystem asset": "Ecosystem asset",
    "Ocean": "Ocean",
    "Species": "Species",
    "Carbon": "Carbon",
    "Urban": "Urban",
    "Protected areas": "Protected areas",
    "Forest": "Forest",
    "No direct SEEA account flag": "No direct SEEA account flag",
}


# ============================================================
# Step 7. Helper functions
# ============================================================

def order_group(df: pd.DataFrame, group_col: str, order: list[str]) -> pd.DataFrame:
    df = df.copy()
    df[group_col] = df[group_col].astype(str)

    return (
        df.set_index(group_col)
        .reindex(order)
        .dropna(how="all")
        .reset_index()
    )


def draw_heatmap(
    ax,
    df: pd.DataFrame,
    group_col: str,
    value_cols: list[str],
    label_map: dict[str, str],
    title: str,
    y_label_map: dict[str, str] | None = None,
    annotate_threshold: int = 15,
) -> mpl.image.AxesImage:
    values = df[value_cols].astype(float)

    image = ax.imshow(
        values,
        vmin=0,
        vmax=100,
        cmap="Blues",
        aspect="auto",
    )

    x_labels = [label_map[col] for col in value_cols]
    y_labels = [str(value) for value in df[group_col]]

    if y_label_map:
        y_labels = [y_label_map.get(value, value) for value in y_labels]

    if "country_count" in df.columns:
        y_labels = [
            f"{label}\n(n={int(count)})"
            for label, count in zip(y_labels, df["country_count"].fillna(0))
        ]

    ax.set_title(title, loc="left", fontsize=TITLE_FS, fontweight="bold", pad=10)

    ax.set_xticks(range(len(value_cols)))
    ax.set_xticklabels(x_labels, fontsize=AXIS_FS, fontweight="bold")

    ax.set_yticks(range(len(y_labels)))
    ax.set_yticklabels(y_labels, fontsize=AXIS_FS, fontweight="bold")

    ax.tick_params(axis="both", length=0, pad=5)

    ax.set_xticks([x - 0.5 for x in range(1, len(value_cols))], minor=True)
    ax.set_yticks([y - 0.5 for y in range(1, len(y_labels))], minor=True)
    ax.grid(which="minor", color="white", linewidth=0.9)
    ax.tick_params(which="minor", bottom=False, left=False)

    for spine in ax.spines.values():
        spine.set_visible(False)

    for y in range(values.shape[0]):
        for x in range(values.shape[1]):
            val = values.iat[y, x]

            if values.shape[1] > annotate_threshold:
                continue

            color = "white" if val >= 55 else "#1f2933"

            ax.text(
                x,
                y,
                f"{val:.1f}",
                ha="center",
                va="center",
                fontsize=CELL_FS,
                fontweight="bold",
                color=color,
            )

    return image


def draw_panel_a(ax) -> mpl.image.AxesImage:
    region = order_group(
        pd.read_csv(PANEL_A_REGION_INPUT),
        "Region",
        REGION_ORDER,
    )

    return draw_heatmap(
        ax,
        region,
        "Region",
        PANEL_A_COLUMNS,
        PANEL_A_LABELS,
        "A. Indicator coverage by region across categories and boundaries",
        REGION_LABELS,
        annotate_threshold=99,
    )


def draw_panel_b(ax) -> mpl.image.AxesImage:
    region = order_group(
        pd.read_csv(PANEL_B_REGION_INPUT),
        "Region",
        REGION_ORDER,
    )

    return draw_heatmap(
        ax,
        region,
        "Region",
        PANEL_B_COLUMNS,
        PANEL_B_LABELS,
        "B. SEEA account coverage by region",
        REGION_LABELS,
    )


def bezier_patch(x0, y0, x1, y1, width, color, zorder=1) -> PathPatch:
    control_dx = (x1 - x0) * 0.52

    verts = [
        (x0, y0 - width / 2),
        (x0 + control_dx, y0 - width / 2),
        (x1 - control_dx, y1 - width / 2),
        (x1, y1 - width / 2),
        (x1, y1 + width / 2),
        (x1 - control_dx, y1 + width / 2),
        (x0 + control_dx, y0 + width / 2),
        (x0, y0 + width / 2),
        (x0, y0 - width / 2),
    ]

    codes = [
        MplPath.MOVETO,
        MplPath.CURVE4,
        MplPath.CURVE4,
        MplPath.CURVE4,
        MplPath.LINETO,
        MplPath.CURVE4,
        MplPath.CURVE4,
        MplPath.CURVE4,
        MplPath.CLOSEPOLY,
    ]

    return PathPatch(
        MplPath(verts, codes),
        facecolor=color,
        edgecolor="none",
        zorder=zorder,
    )


def stack_positions(
    totals: pd.Series,
    low: float,
    high: float,
    gap: float,
    scale: float | None = None,
) -> dict[str, tuple[float, float]]:
    if scale is None:
        total_value = totals.sum()
        available = high - low - gap * (len(totals) - 1)
        scale = available / total_value

    positions = {}
    cursor = high

    for item, value in totals.items():
        height = value * scale
        y1 = cursor
        y0 = cursor - height
        positions[item] = (y0, y1)
        cursor = y0 - gap

    return positions


def load_panel_c_source_total() -> float:
    """
    Calculate n for the Panel C source header.

    n = sum(weighted_indicator_count)
    from step5_source_by_seea_theme_summary.csv
    """
    summary = pd.read_csv(PANEL_C_THEME_SUMMARY_INPUT)

    required_cols = {"dataset", "theme", "weighted_indicator_count"}
    missing_cols = required_cols - set(summary.columns)

    if missing_cols:
        raise ValueError(
            f"{PANEL_C_THEME_SUMMARY_INPUT} is missing required columns: {missing_cols}"
        )

    summary = summary.copy()
    summary["weighted_indicator_count"] = pd.to_numeric(
        summary["weighted_indicator_count"],
        errors="coerce",
    ).fillna(0)

    return float(summary["weighted_indicator_count"].sum())


def ensure_no_account_in_links(links: pd.DataFrame) -> pd.DataFrame:
    """
    Make sure 'No direct SEEA account flag' appears in Panel C links
    when it exists in step5_source_by_seea_theme_summary.csv.

    This protects the combined figure if step5_panel_C_sankey_links.csv
    was created before no_account was added correctly.
    """
    summary = pd.read_csv(PANEL_C_THEME_SUMMARY_INPUT)

    required_cols = {"dataset", "theme", "weighted_indicator_count"}
    missing_cols = required_cols - set(summary.columns)

    if missing_cols:
        raise ValueError(
            f"{PANEL_C_THEME_SUMMARY_INPUT} is missing required columns: {missing_cols}"
        )

    summary = summary.copy()
    summary["weighted_indicator_count"] = pd.to_numeric(
        summary["weighted_indicator_count"],
        errors="coerce",
    ).fillna(0)

    no_account_summary = summary[
        summary["theme"].astype(str).eq("No direct SEEA account flag")
        & summary["weighted_indicator_count"].gt(0)
    ].copy()

    if no_account_summary.empty:
        return links

    existing_no_account = links[
        links["theme"].astype(str).eq("No direct SEEA account flag")
    ]

    existing_pairs = set(
        zip(
            existing_no_account["dataset"].astype(str),
            existing_no_account["theme"].astype(str),
        )
    )

    rows_to_add = []

    for row in no_account_summary.itertuples(index=False):
        dataset = str(row.dataset)
        theme = str(row.theme)

        if (dataset, theme) not in existing_pairs:
            rows_to_add.append(
                {
                    "dataset": dataset,
                    "theme": theme,
                    "flow_value": float(row.weighted_indicator_count),
                    "raw_account_link_count": float(row.weighted_indicator_count),
                }
            )

    if rows_to_add:
        links = pd.concat(
            [links, pd.DataFrame(rows_to_add)],
            ignore_index=True,
        )

    return links


def draw_panel_c(ax) -> None:
    links = pd.read_csv(PANEL_C_LINKS_INPUT)

    required_cols = {"dataset", "theme", "flow_value"}
    missing_cols = required_cols - set(links.columns)

    if missing_cols:
        raise ValueError(
            f"{PANEL_C_LINKS_INPUT} is missing required columns: {missing_cols}"
        )

    # Make sure No direct SEEA account flag is included when available.
    links = ensure_no_account_in_links(links)

    links = links.copy()
    links["flow_value"] = pd.to_numeric(
        links["flow_value"],
        errors="coerce",
    ).fillna(0)

    links = links[links["flow_value"].gt(0)].copy()

    links["dataset"] = pd.Categorical(
        links["dataset"],
        DATASET_ORDER,
        ordered=True,
    )

    links["theme"] = pd.Categorical(
        links["theme"],
        THEME_ORDER,
        ordered=True,
    )

    links = (
        links.dropna(subset=["dataset", "theme"])
        .sort_values(["dataset", "theme"])
        .reset_index(drop=True)
    )

    source_totals = (
        links.groupby("dataset", observed=True)["flow_value"]
        .sum()
        .reindex(DATASET_ORDER)
        .dropna()
    )

    theme_totals = (
        links.groupby("theme", observed=True)["flow_value"]
        .sum()
        .reindex(THEME_ORDER)
        .dropna()
    )

    # n for the left-side label:
    # Data source (n = xxxx)
    panel_c_source_total = load_panel_c_source_total()

    # ------------------------------------------------------------
    # Shared scale for bars and links.
    # This makes the flows fully occupy the colored source bars.
    # ------------------------------------------------------------
    source_low = 0.10
    source_high = 0.88
    source_gap = 0.06

    theme_low = 0.04
    theme_high = 0.90
    theme_gap = 0.014

    total_value = links["flow_value"].sum()

    source_available = source_high - source_low - source_gap * (len(source_totals) - 1)
    theme_available = theme_high - theme_low - theme_gap * (len(theme_totals) - 1)

    source_scale = source_available / total_value
    theme_scale = theme_available / total_value

    flow_scale = min(source_scale, theme_scale)

    source_pos = stack_positions(
        source_totals,
        low=source_low,
        high=source_high,
        gap=source_gap,
        scale=flow_scale,
    )

    theme_pos = stack_positions(
        theme_totals,
        low=theme_low,
        high=theme_high,
        gap=theme_gap,
        scale=flow_scale,
    )

    x_source = 0.045
    x_target = 0.640
    bar_w = 0.018

    # Small overlap prevents a white seam between source bars and flows.
    overlap = 0.0015

    # Start from the top of each bar and stack downward.
    source_offsets = {key: source_pos[key][1] for key in source_pos}
    theme_offsets = {key: theme_pos[key][1] for key in theme_pos}

    # ------------------------------------------------------------
    # Draw Sankey links first.
    # ------------------------------------------------------------
    for row in links.itertuples(index=False):
        dataset = str(row.dataset)
        theme = str(row.theme)
        height = row.flow_value * flow_scale

        src_top = source_offsets[dataset]
        src_bottom = src_top - height
        y0 = (src_top + src_bottom) / 2
        source_offsets[dataset] = src_bottom

        tgt_top = theme_offsets[theme]
        tgt_bottom = tgt_top - height
        y1 = (tgt_top + tgt_bottom) / 2
        theme_offsets[theme] = tgt_bottom

        ax.add_patch(
            bezier_patch(
                x_source + bar_w - overlap,
                y0,
                x_target + overlap,
                y1,
                max(height, 0.0015),
                LINK_COLORS.get(dataset, (0.5, 0.5, 0.5, 0.2)),
                zorder=1,
            )
        )

    # ------------------------------------------------------------
    # Draw source bars on top of links.
    # ------------------------------------------------------------
    for dataset, (y0, y1) in source_pos.items():
        color = DATASET_COLORS.get(str(dataset), "#777777")

        ax.add_patch(
            Rectangle(
                (x_source, y0),
                bar_w,
                y1 - y0,
                facecolor=color,
                edgecolor="white",
                lw=0.4,
                zorder=3,
            )
        )

        ax.text(
            x_source - 0.018,
            (y0 + y1) / 2,
            str(dataset),
            ha="right",
            va="center",
            fontsize=SANK_SOURCE_FS,
            fontweight="bold",
            zorder=4,
        )

        ax.text(
            x_source + bar_w + 0.010,
            (y0 + y1) / 2,
            f"{source_totals[dataset]:.0f}",
            ha="left",
            va="center",
            fontsize=SANK_SOURCE_VALUE_FS,
            fontweight="bold",
            color="#1f2933",
            zorder=4,
        )

    # ------------------------------------------------------------
    # Create right-side label positions.
    # ------------------------------------------------------------
    max_theme = theme_totals.max()
    label_items = list(theme_pos.keys())
    label_y_positions = {}

    if len(label_items) > 1:
        evenly_spaced = list(
            reversed(
                [
                    0.065 + i * (0.835 / (len(label_items) - 1))
                    for i in range(len(label_items))
                ]
            )
        )
        label_y_positions = dict(zip(label_items, evenly_spaced))

    elif label_items:
        label_y_positions[label_items[0]] = 0.50

    # ------------------------------------------------------------
    # Draw target bars and right-side labels.
    # ------------------------------------------------------------
    for theme, (y0, y1) in theme_pos.items():
        face = "#a33a3a" if str(theme) == "No direct SEEA account flag" else "#3a6f96"
        alpha = 0.40 + 0.50 * (theme_totals[theme] / max_theme)

        ax.add_patch(
            Rectangle(
                (x_target, y0),
                bar_w,
                y1 - y0,
                facecolor=face,
                alpha=alpha,
                edgecolor="white",
                lw=0.25,
                zorder=3,
            )
        )

        label = THEME_LABELS.get(str(theme), str(theme))
        label_y = label_y_positions[theme]
        node_y = (y0 + y1) / 2

        ax.plot(
            [x_target + bar_w + 0.004, x_target + 0.058],
            [node_y, label_y],
            color="#94a3b8",
            lw=0.45,
            alpha=0.80,
            zorder=2,
        )

        ax.text(
            x_target + 0.075,
            label_y,
            label,
            ha="left",
            va="center",
            fontsize=SANK_LABEL_FS,
            fontweight="bold",
            zorder=4,
        )

        ax.text(
            x_target - 0.012,
            node_y,
            f"{theme_totals[theme]:.1f}",
            ha="right",
            va="center",
            fontsize=SANK_VALUE_FS,
            fontweight="bold",
            color="#1f2933",
            zorder=4,
        )

    ax.text(
        0.02,
        0.990,
        "C. Indicator links to SEEA thematic groups",
        ha="left",
        va="top",
        fontsize=SANK_TITLE_FS,
        fontweight="bold",
    )

    ax.text(
        x_source,
        0.92,
        f"Data source (n = {panel_c_source_total:.0f}, stock and flow indicators)",
        ha="left",
        va="bottom",
        fontsize=SANK_HEADER_FS,
        fontweight="bold",
        color="#1f2933",
    )

    ax.text(
        x_target,
        0.92,
        "SEEA thematic group",
        ha="center",
        va="bottom",
        fontsize=SANK_HEADER_FS,
        fontweight="bold",
        color="#1f2933",
    )

    ax.text(
        0.02,
        0.01,
        "Flow widths are weighted counts: each indicator contributes one unit, divided equally across linked themes",
        ha="left",
        va="bottom",
        fontsize=NOTE_FS,
        fontweight="bold",
        color="#475569",
    )

    ax.set_xlim(0, 1.15)
    ax.set_ylim(0, 1.0)
    ax.axis("off")



In [1]:
from __future__ import annotations

from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.path import Path as MplPath
from matplotlib.patches import PathPatch, Rectangle


# ============================================================
# Step 1. File names
# ============================================================

PANEL_A_REGION_INPUT = Path("step2_region_category_average_shares.csv")
PANEL_B_REGION_INPUT = Path("step4_region_seea_account_average_coverage.csv")
PANEL_C_LINKS_INPUT = Path("step5_panel_C_sankey_links.csv")

# Used to make sure no_account is included in Panel C
# and to calculate Data source (n = xxxx).
# Required columns: dataset, theme, weighted_indicator_count
PANEL_C_THEME_SUMMARY_INPUT = Path("step5_source_by_seea_theme_summary.csv")

# File used to add indicator counts to Panel A x-axis labels.
# This file must include columns: category and indicator_count.
PANEL_A_COUNT_INPUT = Path("step1_indicator_category_summary.csv")

OUTPUT_AB_PDF = Path("figure_panels_A_B_region_coverage.pdf")
OUTPUT_AB_SVG = Path("figure_panels_A_B_region_coverage.svg")
OUTPUT_AB_PNG = Path("figure_panels_A_B_region_coverage.png")

OUTPUT_C_PDF = Path("figure_panel_C_indicator_seea_links.pdf")
OUTPUT_C_SVG = Path("figure_panel_C_indicator_seea_links.svg")
OUTPUT_C_PNG = Path("figure_panel_C_indicator_seea_links.png")


# ============================================================
# Step 2. Figure settings
# ============================================================

MAX_FIGURE_WIDTH_IN = 6.3
MAX_FIGURE_HEIGHT_IN = 8.0

FIGURE_AB_SIZE = (6.3, 6.8)

# Figure C is now shorter than the original 8.0-inch version.
FIGURE_C_SIZE = (6.3, 6.0)

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["font.family"] = "DejaVu Sans"
mpl.rcParams["axes.unicode_minus"] = False


# ============================================================
# Step 2b. Journal-ready font sizes for 6.3-inch figures
# ============================================================

TITLE_FS = 8.6
AXIS_FS = 5.1
CELL_FS = 5.0

SANK_TITLE_FS = 9.5
SANK_HEADER_FS = 6.2
SANK_SOURCE_FS = 7.2
SANK_SOURCE_VALUE_FS = 6.3
SANK_LABEL_FS = 5.15
SANK_VALUE_FS = 5.3
NOTE_FS = 5.5

CBAR_FS = 6.2


# ============================================================
# Step 3. Region order and labels
# ============================================================

REGION_ORDER = [
    "Eurostat + UK",
    "Non-Eurostat Europe & Central Asia",
    "Middle East, North Africa, Afghanistan & Pakistan",
    "Sub-Saharan Africa",
    "South Asia",
    "East Asia & Pacific",
    "North America",
    "Latin America & Caribbean",
]

REGION_LABELS = {
    "Middle East, North Africa, Afghanistan & Pakistan": "MENA,\nAfghanistan & Pakistan",
    "Latin America & Caribbean": "Latin America &\nCaribbean",
    "Eurostat + UK": "Eurostat + UK",
    "Non-Eurostat Europe & Central Asia": "Non-Eurostat Europe &\nCentral Asia",
    "East Asia & Pacific": "East Asia &\nPacific",
    "Sub-Saharan Africa": "Sub-Saharan\nAfrica",
}


# ============================================================
# Step 4. Panel A columns and labels
# ============================================================

PANEL_A_COLUMNS = [
    "Physical stock",
    "Monetary stock",
    "Physical flow",
    "Monetary flow",
    "Action_kept",
    "Policy Guide_kept",
    "HPS_monetary_only",
    "DDP_monetary_only",
    "MS-NMS_monetary_only",
    "ADE_monetary_only",
]

PANEL_A_BASE_LABELS = {
    "Physical stock": "Physical\nstock",
    "Monetary stock": "Monetary\nstock",
    "Physical flow": "Physical\nflow",
    "Monetary flow": "Monetary\nflow",
    "Action_kept": "Action",
    "Policy Guide_kept": "Policy\nguide",
    "HPS_monetary_only": "Monetary\nHPS",
    "DDP_monetary_only": "Monetary\nDDP",
    "MS-NMS_monetary_only": "Monetary\nMS-NMS",
    "ADE_monetary_only": "Monetary\nADE",
}


def make_panel_a_labels() -> dict[str, str]:
    counts = pd.read_csv(PANEL_A_COUNT_INPUT)

    required_cols = {"category", "indicator_count"}
    missing_cols = required_cols - set(counts.columns)

    if missing_cols:
        raise ValueError(
            f"{PANEL_A_COUNT_INPUT} is missing required columns: {missing_cols}"
        )

    count_map = dict(
        zip(
            counts["category"].astype(str),
            counts["indicator_count"].astype(int),
        )
    )

    return {
        col: f"{PANEL_A_BASE_LABELS[col]}\n(n={count_map.get(col, 0)})"
        for col in PANEL_A_COLUMNS
    }


# ============================================================
# Step 5. Panel B columns and labels
# ============================================================

PANEL_B_COLUMNS = [
    "Reported SEEA accounts",
    "Easily accessible accounts",
    "Found but not reported",
]

PANEL_B_LABELS = {
    "Reported SEEA accounts": "Reported\nSEEA accounts",
    "Easily accessible accounts": "Easily accessible\naccounts",
    "Found but not\nreported": "Account found but not\nreported",
    "Found but not reported": "Account found but not\nreported",
}


# ============================================================
# Step 6. Panel C Sankey settings
# ============================================================

DATASET_ORDER = ["WB", "IMF", "UNSD"]

DATASET_COLORS = {
    "WB": "#b2182b",
    "IMF": "#238b45",
    "UNSD": "#2166ac",
}

LINK_COLORS = {
    "WB": (178 / 255, 24 / 255, 43 / 255, 0.26),
    "IMF": (35 / 255, 139 / 255, 69 / 255, 0.24),
    "UNSD": (33 / 255, 102 / 255, 172 / 255, 0.25),
}

THEME_ORDER = [
    "Material flow",
    "Water",
    "Energy",
    "Mineral and energy resources",
    "Air emissions",
    "Land",
    "Timber resources",
    "Aquatic resources",
    "Other biological resources",
    "Agriculture, forestry and fisheries",
    "Waste",
    "Environmental protection expenditure",
    "Resource management expenditure",
    "Environmental goods and services",
    "Environmental taxes",
    "Environmental subsidies and similar transfers",
    "Ecosystem extent",
    "Ecosystem condition",
    "Ecosystem services",
    "Ecosystem asset",
    "Ocean",
    "Species",
    "Carbon",
    "Urban",
    "Protected areas",
    "Forest",
    "No direct SEEA account flag",
]

THEME_LABELS = {
    "Material flow": "Material flow",
    "Water": "Water",
    "Energy": "Energy",
    "Mineral and energy resources": "Mineral and energy\nresources",
    "Air emissions": "Air emissions",
    "Land": "Land",
    "Timber resources": "Timber resources",
    "Aquatic resources": "Aquatic resources",
    "Other biological resources": "Other biological\nresources",
    "Agriculture, forestry and fisheries": "Agriculture, forestry\nand fisheries",
    "Waste": "Waste",
    "Environmental protection expenditure": "Environmental protection\nexpenditure",
    "Resource management expenditure": "Resource management\nexpenditure",
    "Environmental goods and services": "Environmental goods\nand services",
    "Environmental taxes": "Environmental taxes",
    "Environmental subsidies and similar transfers": "Environmental subsidies\nand similar transfers",
    "Ecosystem extent": "Ecosystem extent",
    "Ecosystem condition": "Ecosystem condition",
    "Ecosystem services": "Ecosystem services",
    "Ecosystem asset": "Ecosystem asset",
    "Ocean": "Ocean",
    "Species": "Species",
    "Carbon": "Carbon",
    "Urban": "Urban",
    "Protected areas": "Protected areas",
    "Forest": "Forest",
    "No direct SEEA account flag": "No direct SEEA\naccount flag",
}


# ============================================================
# Step 7. Helper functions
# ============================================================

def order_group(df: pd.DataFrame, group_col: str, order: list[str]) -> pd.DataFrame:
    df = df.copy()
    df[group_col] = df[group_col].astype(str)

    return (
        df.set_index(group_col)
        .reindex(order)
        .dropna(how="all")
        .reset_index()
    )


def draw_heatmap(
    ax,
    df: pd.DataFrame,
    group_col: str,
    value_cols: list[str],
    label_map: dict[str, str],
    title: str,
    y_label_map: dict[str, str] | None = None,
    annotate_threshold: int = 15,
) -> mpl.image.AxesImage:
    values = df[value_cols].astype(float)

    image = ax.imshow(
        values,
        vmin=0,
        vmax=100,
        cmap="Blues",
        aspect="auto",
    )

    x_labels = [label_map[col] for col in value_cols]
    y_labels = [str(value) for value in df[group_col]]

    if y_label_map:
        y_labels = [y_label_map.get(value, value) for value in y_labels]

    if "country_count" in df.columns:
        y_labels = [
            f"{label}\n(n={int(count)})"
            for label, count in zip(y_labels, df["country_count"].fillna(0))
        ]

    ax.set_title(title, loc="left", fontsize=TITLE_FS, fontweight="bold", pad=7)

    ax.set_xticks(range(len(value_cols)))
    ax.set_xticklabels(x_labels, fontsize=AXIS_FS, fontweight="bold")

    ax.set_yticks(range(len(y_labels)))
    ax.set_yticklabels(y_labels, fontsize=AXIS_FS, fontweight="bold")

    ax.tick_params(axis="both", length=0, pad=3)

    ax.set_xticks([x - 0.5 for x in range(1, len(value_cols))], minor=True)
    ax.set_yticks([y - 0.5 for y in range(1, len(y_labels))], minor=True)
    ax.grid(which="minor", color="white", linewidth=0.55)
    ax.tick_params(which="minor", bottom=False, left=False)

    for spine in ax.spines.values():
        spine.set_visible(False)

    for y in range(values.shape[0]):
        for x in range(values.shape[1]):
            val = values.iat[y, x]

            if values.shape[1] > annotate_threshold:
                continue

            color = "white" if val >= 55 else "#1f2933"

            ax.text(
                x,
                y,
                f"{val:.1f}",
                ha="center",
                va="center",
                fontsize=CELL_FS,
                fontweight="bold",
                color=color,
            )

    return image


def draw_panel_a(ax) -> mpl.image.AxesImage:
    region = order_group(
        pd.read_csv(PANEL_A_REGION_INPUT),
        "Region",
        REGION_ORDER,
    )

    panel_a_labels = make_panel_a_labels()

    return draw_heatmap(
        ax,
        region,
        "Region",
        PANEL_A_COLUMNS,
        panel_a_labels,
        "A. Indicator coverage by region\nacross categories and boundaries",
        REGION_LABELS,
        annotate_threshold=99,
    )


def draw_panel_b(ax) -> mpl.image.AxesImage:
    region = order_group(
        pd.read_csv(PANEL_B_REGION_INPUT),
        "Region",
        REGION_ORDER,
    )

    return draw_heatmap(
        ax,
        region,
        "Region",
        PANEL_B_COLUMNS,
        PANEL_B_LABELS,
        "B. SEEA account coverage by region",
        REGION_LABELS,
    )


def bezier_patch(x0, y0, x1, y1, width, color, zorder=1) -> PathPatch:
    control_dx = (x1 - x0) * 0.52

    verts = [
        (x0, y0 - width / 2),
        (x0 + control_dx, y0 - width / 2),
        (x1 - control_dx, y1 - width / 2),
        (x1, y1 - width / 2),
        (x1, y1 + width / 2),
        (x1 - control_dx, y1 + width / 2),
        (x0 + control_dx, y0 + width / 2),
        (x0, y0 + width / 2),
        (x0, y0 - width / 2),
    ]

    codes = [
        MplPath.MOVETO,
        MplPath.CURVE4,
        MplPath.CURVE4,
        MplPath.CURVE4,
        MplPath.LINETO,
        MplPath.CURVE4,
        MplPath.CURVE4,
        MplPath.CURVE4,
        MplPath.CLOSEPOLY,
    ]

    return PathPatch(
        MplPath(verts, codes),
        facecolor=color,
        edgecolor="none",
        zorder=zorder,
    )


def stack_positions(
    totals: pd.Series,
    low: float,
    high: float,
    gap: float,
    scale: float | None = None,
) -> dict[str, tuple[float, float]]:
    if scale is None:
        total_value = totals.sum()
        available = high - low - gap * (len(totals) - 1)
        scale = available / total_value

    positions = {}
    cursor = high

    for item, value in totals.items():
        height = value * scale
        y1 = cursor
        y0 = cursor - height
        positions[item] = (y0, y1)
        cursor = y0 - gap

    return positions


def load_panel_c_source_total() -> float:
    summary = pd.read_csv(PANEL_C_THEME_SUMMARY_INPUT)

    required_cols = {"dataset", "theme", "weighted_indicator_count"}
    missing_cols = required_cols - set(summary.columns)

    if missing_cols:
        raise ValueError(
            f"{PANEL_C_THEME_SUMMARY_INPUT} is missing required columns: {missing_cols}"
        )

    summary = summary.copy()
    summary["weighted_indicator_count"] = pd.to_numeric(
        summary["weighted_indicator_count"],
        errors="coerce",
    ).fillna(0)

    return float(summary["weighted_indicator_count"].sum())


def ensure_no_account_in_links(links: pd.DataFrame) -> pd.DataFrame:
    summary = pd.read_csv(PANEL_C_THEME_SUMMARY_INPUT)

    required_cols = {"dataset", "theme", "weighted_indicator_count"}
    missing_cols = required_cols - set(summary.columns)

    if missing_cols:
        raise ValueError(
            f"{PANEL_C_THEME_SUMMARY_INPUT} is missing required columns: {missing_cols}"
        )

    summary = summary.copy()
    summary["weighted_indicator_count"] = pd.to_numeric(
        summary["weighted_indicator_count"],
        errors="coerce",
    ).fillna(0)

    no_account_summary = summary[
        summary["theme"].astype(str).eq("No direct SEEA account flag")
        & summary["weighted_indicator_count"].gt(0)
    ].copy()

    if no_account_summary.empty:
        return links

    existing_no_account = links[
        links["theme"].astype(str).eq("No direct SEEA account flag")
    ]

    existing_pairs = set(
        zip(
            existing_no_account["dataset"].astype(str),
            existing_no_account["theme"].astype(str),
        )
    )

    rows_to_add = []

    for row in no_account_summary.itertuples(index=False):
        dataset = str(row.dataset)
        theme = str(row.theme)

        if (dataset, theme) not in existing_pairs:
            rows_to_add.append(
                {
                    "dataset": dataset,
                    "theme": theme,
                    "flow_value": float(row.weighted_indicator_count),
                    "raw_account_link_count": float(row.weighted_indicator_count),
                }
            )

    if rows_to_add:
        links = pd.concat(
            [links, pd.DataFrame(rows_to_add)],
            ignore_index=True,
        )

    return links


# ============================================================
# Step 8. Panel C Sankey
# ============================================================

def draw_panel_c(ax) -> None:
    links = pd.read_csv(PANEL_C_LINKS_INPUT)

    required_cols = {"dataset", "theme", "flow_value"}
    missing_cols = required_cols - set(links.columns)

    if missing_cols:
        raise ValueError(
            f"{PANEL_C_LINKS_INPUT} is missing required columns: {missing_cols}"
        )

    links = ensure_no_account_in_links(links)

    links = links.copy()
    links["flow_value"] = pd.to_numeric(
        links["flow_value"],
        errors="coerce",
    ).fillna(0)

    links = links[links["flow_value"].gt(0)].copy()

    links["dataset"] = pd.Categorical(
        links["dataset"],
        DATASET_ORDER,
        ordered=True,
    )

    links["theme"] = pd.Categorical(
        links["theme"],
        THEME_ORDER,
        ordered=True,
    )

    links = (
        links.dropna(subset=["dataset", "theme"])
        .sort_values(["dataset", "theme"])
        .reset_index(drop=True)
    )

    source_totals = (
        links.groupby("dataset", observed=True)["flow_value"]
        .sum()
        .reindex(DATASET_ORDER)
        .dropna()
    )

    theme_totals = (
        links.groupby("theme", observed=True)["flow_value"]
        .sum()
        .reindex(THEME_ORDER)
        .dropna()
    )

    panel_c_source_total = load_panel_c_source_total()

    # Vertical layout.
    source_low = 0.105
    source_high = 0.875
    source_gap = 0.045

    theme_low = 0.055
    theme_high = 0.895
    theme_gap = 0.009

    total_value = links["flow_value"].sum()

    source_available = source_high - source_low - source_gap * (len(source_totals) - 1)
    theme_available = theme_high - theme_low - theme_gap * (len(theme_totals) - 1)

    source_scale = source_available / total_value
    theme_scale = theme_available / total_value
    flow_scale = min(source_scale, theme_scale)

    source_pos = stack_positions(
        source_totals,
        low=source_low,
        high=source_high,
        gap=source_gap,
        scale=flow_scale,
    )

    theme_pos = stack_positions(
        theme_totals,
        low=theme_low,
        high=theme_high,
        gap=theme_gap,
        scale=flow_scale,
    )

    # Horizontal layout.
    # Increasing x_target moves the SEEA thematic group bars to the right.
    x_source = 0.055
    x_target = 0.665
    bar_w = 0.018
    overlap = 0.0015

    # Right-side labels and connector lines.
    label_line_x0 = x_target + bar_w + 0.006
    label_line_x1 = x_target + 0.185
    label_text_x = x_target + 0.200
    value_text_x = x_target - 0.012
    theme_header_x = x_target + 0.095

    source_offsets = {key: source_pos[key][1] for key in source_pos}
    theme_offsets = {key: theme_pos[key][1] for key in theme_pos}

    for row in links.itertuples(index=False):
        dataset = str(row.dataset)
        theme = str(row.theme)
        height = row.flow_value * flow_scale

        src_top = source_offsets[dataset]
        src_bottom = src_top - height
        y0 = (src_top + src_bottom) / 2
        source_offsets[dataset] = src_bottom

        tgt_top = theme_offsets[theme]
        tgt_bottom = tgt_top - height
        y1 = (tgt_top + tgt_bottom) / 2
        theme_offsets[theme] = tgt_bottom

        ax.add_patch(
            bezier_patch(
                x_source + bar_w - overlap,
                y0,
                x_target + overlap,
                y1,
                max(height, 0.0015),
                LINK_COLORS.get(dataset, (0.5, 0.5, 0.5, 0.2)),
                zorder=1,
            )
        )

    for dataset, (y0, y1) in source_pos.items():
        color = DATASET_COLORS.get(str(dataset), "#777777")

        ax.add_patch(
            Rectangle(
                (x_source, y0),
                bar_w,
                y1 - y0,
                facecolor=color,
                edgecolor="white",
                lw=0.35,
                zorder=3,
            )
        )

        ax.text(
            x_source - 0.018,
            (y0 + y1) / 2,
            str(dataset),
            ha="right",
            va="center",
            fontsize=SANK_SOURCE_FS,
            fontweight="bold",
            zorder=4,
        )

        ax.text(
            x_source + bar_w + 0.010,
            (y0 + y1) / 2,
            f"{source_totals[dataset]:.0f}",
            ha="left",
            va="center",
            fontsize=SANK_SOURCE_VALUE_FS,
            fontweight="bold",
            color="#1f2933",
            zorder=4,
        )

    max_theme = theme_totals.max()
    label_items = list(theme_pos.keys())
    label_y_positions = {}

    if len(label_items) > 1:
        evenly_spaced = list(
            reversed(
                [
                    0.065 + i * (0.820 / (len(label_items) - 1))
                    for i in range(len(label_items))
                ]
            )
        )
        label_y_positions = dict(zip(label_items, evenly_spaced))
    elif label_items:
        label_y_positions[label_items[0]] = 0.50

    for theme, (y0, y1) in theme_pos.items():
        face = "#a33a3a" if str(theme) == "No direct SEEA account flag" else "#3a6f96"
        alpha = 0.40 + 0.50 * (theme_totals[theme] / max_theme)

        ax.add_patch(
            Rectangle(
                (x_target, y0),
                bar_w,
                y1 - y0,
                facecolor=face,
                alpha=alpha,
                edgecolor="white",
                lw=0.22,
                zorder=3,
            )
        )

        label = THEME_LABELS.get(str(theme), str(theme))
        label_y = label_y_positions[theme]
        node_y = (y0 + y1) / 2

        ax.plot(
            [label_line_x0, label_line_x1],
            [node_y, label_y],
            color="#94a3b8",
            lw=0.38,
            alpha=0.80,
            zorder=2,
        )

        ax.text(
            label_text_x,
            label_y,
            label,
            ha="left",
            va="center",
            fontsize=SANK_LABEL_FS,
            fontweight="bold",
            zorder=4,
        )

        ax.text(
            value_text_x,
            node_y,
            f"{theme_totals[theme]:.1f}",
            ha="right",
            va="center",
            fontsize=SANK_VALUE_FS,
            fontweight="bold",
            color="#1f2933",
            zorder=4,
        )



    ax.text(
        x_source,
        0.910,
        f"Data source\n(n = {panel_c_source_total:.0f}, stock and flow indicators)",
        ha="left",
        va="bottom",
        fontsize=SANK_HEADER_FS,
        fontweight="bold",
        color="#1f2933",
    )

    ax.text(
        theme_header_x,
        0.918,
        "SEEA thematic group",
        ha="center",
        va="bottom",
        fontsize=SANK_HEADER_FS,
        fontweight="bold",
        color="#1f2933",
    )

    ax.text(
        0.015,
        0.018,
        "Flow widths are weighted counts: each indicator contributes one unit, divided equally across linked themes",
        ha="left",
        va="bottom",
        fontsize=NOTE_FS,
        fontweight="bold",
        color="#475569",
    )

    ax.set_xlim(0, 1.18)
    ax.set_ylim(0, 1.0)
    ax.axis("off")


# ============================================================
# Step 9. Save and create figures
# ============================================================

def save_figure(fig: mpl.figure.Figure, pdf_path: Path, svg_path: Path, png_path: Path) -> None:
    width, height = fig.get_size_inches()

    if width > MAX_FIGURE_WIDTH_IN or height > MAX_FIGURE_HEIGHT_IN:
        raise ValueError(
            f"Figure size {width:.2f} x {height:.2f} inches exceeds "
            f"{MAX_FIGURE_WIDTH_IN} x {MAX_FIGURE_HEIGHT_IN} inches."
        )

    fig.savefig(pdf_path)
    fig.savefig(svg_path)
    fig.savefig(png_path, dpi=300)


def create_figure_ab() -> None:
    fig = plt.figure(figsize=FIGURE_AB_SIZE, constrained_layout=False)

    outer = fig.add_gridspec(
        nrows=2,
        ncols=1,
        height_ratios=[1.08, 1.0],
        hspace=0.30,
        top=0.945,
        bottom=0.075,
        left=0.190,
        right=0.890,
    )

    ax_a = fig.add_subplot(outer[0])
    ax_b = fig.add_subplot(outer[1])

    image = draw_panel_a(ax_a)
    draw_panel_b(ax_b)

    pos_a = ax_a.get_position()
    pos_b = ax_b.get_position()

    cax = fig.add_axes([0.920, pos_b.y0, 0.018, pos_a.y1 - pos_b.y0])
    cbar = fig.colorbar(image, cax=cax)

    cbar.ax.tick_params(labelsize=CBAR_FS, length=2, pad=2)

    for tick in cbar.ax.get_yticklabels():
        tick.set_fontweight("bold")

    cbar.set_label(
        "Average country coverage (%)",
        fontsize=CBAR_FS,
        fontweight="bold",
        labelpad=1.5,
    )

    save_figure(fig, OUTPUT_AB_PDF, OUTPUT_AB_SVG, OUTPUT_AB_PNG)
    plt.close(fig)


def create_figure_c() -> None:
    fig = plt.figure(figsize=FIGURE_C_SIZE, constrained_layout=False)

    # Use almost the full canvas width for Panel C.
    # Format: [left, bottom, width, height]
    ax_sankey = fig.add_axes([0.035, 0.035, 0.960, 0.935])

    draw_panel_c(ax_sankey)

    save_figure(fig, OUTPUT_C_PDF, OUTPUT_C_SVG, OUTPUT_C_PNG)
    plt.close(fig)


def main() -> None:
    create_figure_ab()
    create_figure_c()

    print("Split figure development complete")
    print(f"Panels A and B PDF: {OUTPUT_AB_PDF}")
    print(f"Panels A and B SVG: {OUTPUT_AB_SVG}")
    print(f"Panels A and B PNG: {OUTPUT_AB_PNG}")
    print(f"Panel C PDF: {OUTPUT_C_PDF}")
    print(f"Panel C SVG: {OUTPUT_C_SVG}")
    print(f"Panel C PNG: {OUTPUT_C_PNG}")


if __name__ == "__main__":
    main()

Split figure development complete
Panels A and B PDF: figure_panels_A_B_region_coverage.pdf
Panels A and B SVG: figure_panels_A_B_region_coverage.svg
Panels A and B PNG: figure_panels_A_B_region_coverage.png
Panel C PDF: figure_panel_C_indicator_seea_links.pdf
Panel C SVG: figure_panel_C_indicator_seea_links.svg
Panel C PNG: figure_panel_C_indicator_seea_links.png
